In [0]:
%pip install openpyxl

import pandas as pd

BASE = "/Volumes/mvp-base_programação_operacional/bronze/raw_extraction"

In [0]:
spark.sql("USE CATALOG `mvp-base_programação_operacional`")

df_silver = spark.table("silver.agendamentos_unificado").toPandas()

print(df_silver.shape)
df_silver.head()

In [0]:
dim_territorio = (
    df_silver[["territorio_normalizado"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_territorio.insert(0, "sk_territorio", range(1, len(dim_territorio) + 1))
dim_territorio.rename(columns={"territorio_normalizado": "territorio"}, inplace=True)

dim_tipo_atividade = (
    df_silver[["tipo_atividade_normalizado"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_tipo_atividade.insert(0, "sk_tipo_atividade", range(1, len(dim_tipo_atividade) + 1))
dim_tipo_atividade.rename(columns={"tipo_atividade_normalizado": "tipo_atividade"}, inplace=True)

dim_origem_sistema = (
    df_silver[["sistema_origem"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_origem_sistema.insert(0, "sk_origem_sistema", range(1, len(dim_origem_sistema) + 1))

print("dim_territorio:", dim_territorio.shape)
print("dim_tipo_atividade:", dim_tipo_atividade.shape)
print("dim_origem_sistema:", dim_origem_sistema.shape)

In [0]:
df_silver["data_str"] = pd.to_datetime(df_silver["data_agendamento"], errors="coerce").dt.strftime("%Y-%m-%d")
df_silver["data_str"] = df_silver["data_str"].fillna("SEM_DATA")

dim_tempo = (
    df_silver[df_silver["data_str"] != "SEM_DATA"][["data_str"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_tempo.insert(0, "sk_tempo", range(1, len(dim_tempo) + 1))
dim_tempo["data_agendamento"] = pd.to_datetime(dim_tempo["data_str"])
dim_tempo["ano"] = dim_tempo["data_agendamento"].dt.year
dim_tempo["mes"] = dim_tempo["data_agendamento"].dt.month
dim_tempo["dia_semana"] = dim_tempo["data_agendamento"].dt.day_name()

linha_sentinela = pd.DataFrame({
    "sk_tempo": [-1],
    "data_str": ["SEM_DATA"],
    "data_agendamento": [pd.NaT],
    "ano": [None],
    "mes": [None],
    "dia_semana": ["Não aplicável"],
})
dim_tempo = pd.concat([dim_tempo, linha_sentinela], ignore_index=True)

print("dim_tempo:", dim_tempo.shape)
dim_tempo.tail()

In [0]:
mapa_data_para_sk = dict(zip(dim_tempo["data_str"], dim_tempo["sk_tempo"]))

fato_agendamento = df_silver.merge(
    dim_territorio, left_on="territorio_normalizado", right_on="territorio", how="left"
).merge(
     dim_tipo_atividade, left_on="tipo_atividade_normalizado", right_on="tipo_atividade", how="left"
).merge(
    dim_origem_sistema, on="sistema_origem", how="left"
)

fato_agendamento["sk_tempo"] = fato_agendamento["data_str"].map(mapa_data_para_sk)

fato_agendamento.insert(0, "sk_agendamento", range(1, len(fato_agendamento) + 1))

fato_agendamento = fato_agendamento[[
    "sk_agendamento", "id_contrato", "sk_territorio", "sk_tempo",
    "sk_tipo_atividade", "sk_origem_sistema", "documento", "tipo_documento",
    "status_unificado", "valor_recorrente", "flag_replicado_multissistema",
]]

print("Total de linhas na fato:", len(fato_agendamento))
print("Linhas sem território mapeado:", fato_agendamento["sk_territorio"].isna().sum())
print("Linhas com sk_tempo = -1 (sem data):", (fato_agendamento["sk_tempo"] == -1).sum())
fato_agendamento.head(10)

In [0]:
spark.createDataFrame(dim_territorio).write.mode("overwrite").saveAsTable("gold.dim_territorio")
spark.createDataFrame(dim_tipo_atividade).write.mode("overwrite").saveAsTable("gold.dim_tipo_atividade")
spark.createDataFrame(dim_origem_sistema).write.mode("overwrite").saveAsTable("gold.dim_origem_sistema")
spark.createDataFrame(dim_tempo.drop(columns=["data_str"])).write.mode("overwrite").saveAsTable("gold.dim_tempo")
spark.createDataFrame(fato_agendamento).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fato_agendamento")

print("5 tabelas gravadas em gold")